# AS-VA-ANP benchmark (Abhilash's code), cell by cell

The code below is the version sent on 7 Nov 2026, split into cells and otherwise **unchanged**. The only fix: the
last lines read `if _name_ == "_main_":` (WhatsApp removed the double underscores), which raises a NameError; here
the benchmark is called directly in its own cell.

Run the cells top to bottom (Runtime → Run all in Colab). Plain NumPy, no GPU needed; about a minute.

After the benchmark, a **Checks** section reruns the identical instances to test what the "adaptive" part adds.

In [1]:
import numpy as np
from numpy.linalg import norm
import time
import matplotlib.pyplot as plt

np.random.seed(42)

## Ml-Kem (Fips 203) Cbd_Eta Noise & Module-Lwe Lattice Generator

In [2]:
# ============================================================================
# 1. ML-KEM (FIPS 203) CBD_eta NOISE & MODULE-LWE LATTICE GENERATOR
# ============================================================================

def sample_cbd(shape, eta=2):
    """Exact Centered Binomial Distribution B_eta from FIPS 203 (ML-KEM)."""
    bits_a = np.random.randint(0, 2, size=(*shape, eta))
    bits_b = np.random.randint(0, 2, size=(*shape, eta))
    return np.sum(bits_a, axis=-1) - np.sum(bits_b, axis=-1)


def negacyclic_matrix(poly_coeffs):
    """Construct the d x d negacyclic matrix for multiplication in Z_q[X]/(X^d + 1)."""
    d = len(poly_coeffs)
    idx = np.arange(d)
    diff = idx[:, None] - idx[None, :]
    M = poly_coeffs[np.mod(diff, d)].astype(float)
    M[diff < 0] *= -1.0
    return M


def generate_mlkem_bdd_basis(k_rank=2, poly_deg=4, q=3329):
    """Generate block-negacyclic q-ary lattice basis for Module-LWE."""
    m = k_rank * poly_deg
    n = 2 * m
    A_blocks = np.zeros((m, m), dtype=float)
    for r in range(k_rank):
        for c in range(k_rank):
            poly_a = np.random.randint(0, q, size=poly_deg)
            A_blocks[r*poly_deg:(r+1)*poly_deg, c*poly_deg:(c+1)*poly_deg] = negacyclic_matrix(poly_a)

    B = np.zeros((n, n), dtype=float)
    B[:m, :m] = q * np.eye(m)
    B[m:, :m] = A_blocks
    B[m:, m:] = np.eye(m)
    return B


def generate_cbd_bdd_target(B_reduced, nrm_sq, eta=2, ratio=0.28):
    """Generate BDD target t = v + e with realistic noise scaling."""
    n = B_reduced.shape[0]
    coeffs = np.random.randint(-3, 4, size=n)
    true_lattice_pt = coeffs @ B_reduced

    gs_norms = np.sqrt(np.maximum(nrm_sq, 1e-12))
    bottleneck_scale = 1.0 / np.mean(1.0 / gs_norms)
    desired_sigma = ratio * bottleneck_scale

    raw_cbd = sample_cbd((n,), eta=eta).astype(float)
    fine_cbd = sample_cbd((n,), eta=eta).astype(float) * 0.25
    unit_noise = (raw_cbd + fine_cbd) / np.sqrt((eta / 2.0) * 1.0625)

    error_vec = desired_sigma * unit_noise
    target = true_lattice_pt + error_vec
    return target, true_lattice_pt, desired_sigma

## Fast Incremental Lll Reduction

In [3]:
# ============================================================================
# 2. FAST INCREMENTAL LLL REDUCTION
# ============================================================================

def gram_schmidt(B):
    """Exact Gram-Schmidt orthogonalization."""
    n = B.shape[0]
    B_star = np.zeros_like(B, dtype=float)
    mu = np.zeros((n, n), dtype=float)
    nrm_sq = np.zeros(n, dtype=float)

    for i in range(n):
        v = B[i].copy()
        for j in range(i):
            if nrm_sq[j] > 1e-12:
                mu[i, j] = np.dot(B[i], B_star[j]) / nrm_sq[j]
                v -= mu[i, j] * B_star[j]
        B_star[i] = v
        nrm_sq[i] = np.dot(v, v)
    return B_star, mu, nrm_sq


def fast_lll_reduce(B, delta=0.99):
    """Fast LLL reduction with incremental mu updates."""
    n = B.shape[0]
    B = B.astype(float).copy()
    B_star, mu, nrm_sq = gram_schmidt(B)

    k = 1
    while k < n:
        for j in range(k - 1, -1, -1):
            if abs(mu[k, j]) > 0.5:
                q_int = round(mu[k, j])
                B[k] -= q_int * B[j]
                mu[k, :j] -= q_int * mu[j, :j]
                mu[k, j] -= q_int

        if nrm_sq[k] >= (delta - mu[k, k - 1] ** 2) * nrm_sq[k - 1]:
            k += 1
        else:
            B[[k, k - 1]] = B[[k - 1, k]]
            B_star, mu, nrm_sq = gram_schmidt(B)
            k = max(k - 1, 1)

    return B, B_star, mu, nrm_sq


def schnorr_euchner_planes(c, num_planes):
    """Exact Schnorr-Euchner zig-zag integer ordering around c."""
    c_round = int(round(c))
    if num_planes == 1:
        return [c_round]
    frac = c - c_round
    step = 1 if frac >= 0 else -1
    if num_planes == 2:
        return [c_round, c_round + step]
    return [c_round, c_round + step, c_round - step]

## Baseline Algorithms

In [4]:
# ============================================================================
# 3. BASELINE ALGORITHMS
# ============================================================================

def babai_nearest_plane(B, B_star, nrm_sq, target):
    """1. Babai's Nearest Plane."""
    start = time.perf_counter()
    n = B.shape[0]
    b = target.copy()
    for i in range(n - 1, -1, -1):
        c = np.dot(b, B_star[i]) / nrm_sq[i]
        b -= round(c) * B[i]
    return target - b, time.perf_counter() - start, {"nodes": n, "children": n}


def lindner_peikert_anp(B, B_star, nrm_sq, target, sigma, max_list=None):
    """2. Lindner-Peikert ANP."""
    start = time.perf_counter()
    n = B.shape[0]
    if max_list is None:
        max_list = 2 * n

    gs_lengths = np.sqrt(np.maximum(nrm_sq, 1e-12))
    med_gs = np.median(gs_lengths)
    candidates = [(target.copy(), 0.0)]
    stats = {"nodes": 0, "children": 0}

    for i in range(n - 1, -1, -1):
        b_star_i = B_star[i]
        nsq = nrm_sq[i]
        d_i = int(np.clip(np.ceil(1.6 * med_gs / gs_lengths[i]), 1, 3))
        new_candidates = []

        for b, acc_dist in candidates:
            stats["nodes"] += 1
            c = np.dot(b, b_star_i) / nsq
            for m_k in schnorr_euchner_planes(c, d_i):
                stats["children"] += 1
                dk = ((c - m_k) ** 2) * nsq
                new_candidates.append((b - m_k * B[i], acc_dist + dk))

        if len(new_candidates) > max_list:
            new_candidates.sort(key=lambda x: x[1])
            candidates = new_candidates[:max_list]
        else:
            candidates = new_candidates

    best_b, _ = min(candidates, key=lambda x: np.dot(x[0], x[0]))
    return target - best_b, time.perf_counter() - start, stats


def discrete_pruned_enumeration_2023(B, B_star, nrm_sq, target, max_cells=None):
    """3. Discrete Pruned Enumeration (2023)."""
    start = time.perf_counter()
    n = B.shape[0]
    if max_cells is None:
        max_cells = 2 * n

    tags = [np.zeros(n, dtype=int)]
    sorted_dims = np.argsort(nrm_sq)

    for idx in sorted_dims:
        tag = np.zeros(n, dtype=int)
        tag[idx] = 1
        tags.append(tag)

    top_k = min(n, int(np.ceil(np.sqrt(2 * max_cells))))
    for a in range(top_k):
        for b_idx in range(a + 1, top_k):
            if len(tags) >= max_cells:
                break
            tag = np.zeros(n, dtype=int)
            tag[sorted_dims[a]] = 1
            tag[sorted_dims[b_idx]] = 1
            tags.append(tag)
        if len(tags) >= max_cells:
            break

    tags = tags[:max_cells]
    best_vec = None
    best_dist_sq = float("inf")
    stats = {"nodes": len(tags) * n, "children": len(tags) * n}

    for tag in tags:
        b = target.copy()
        for i in range(n - 1, -1, -1):
            c = np.dot(b, B_star[i]) / nrm_sq[i]
            se = schnorr_euchner_planes(c, tag[i] + 1)
            b -= se[tag[i]] * B[i]
        dist_sq = np.dot(b, b)
        if dist_sq < best_dist_sq:
            best_dist_sq = dist_sq
            best_vec = target - b

    return best_vec, time.perf_counter() - start, stats


def static_va_anp_heuristic(B, B_star, nrm_sq, target, sigma,
                            llr_threshold=2.2, sphere_mult=2.8, max_list=None):
    """4. Static VA-ANP (Heuristic Baseline)."""
    start = time.perf_counter()
    n = B.shape[0]
    if max_list is None:
        max_list = 2 * n

    max_penalty = llr_threshold * 2.0 * (sigma ** 2)
    sphere_radius_sq = sphere_mult * n * (sigma ** 2)
    candidates = [(target.copy(), 0.0)]
    stats = {"nodes": 0, "children": 0}

    for i in range(n - 1, -1, -1):
        b_star_i = B_star[i]
        nsq = nrm_sq[i]
        new_candidates = []

        for b, acc_dist in candidates:
            stats["nodes"] += 1
            c = np.dot(b, b_star_i) / nsq
            se_planes = schnorr_euchner_planes(c, 3)
            d0 = ((c - se_planes[0]) ** 2) * nsq

            if acc_dist + d0 <= sphere_radius_sq or len(new_candidates) == 0:
                new_candidates.append((b - se_planes[0] * B[i], acc_dist + d0))
                stats["children"] += 1

            for m_k in se_planes[1:]:
                dk = ((c - m_k) ** 2) * nsq
                if (dk - d0) <= max_penalty and (acc_dist + dk) <= sphere_radius_sq:
                    new_candidates.append((b - m_k * B[i], acc_dist + dk))
                    stats["children"] += 1
                else:
                    break

        if not new_candidates:
            b, acc_dist = candidates[0]
            c = np.dot(b, B_star[i]) / nsq
            m0 = round(c)
            d0 = ((c - m0) ** 2) * nsq
            new_candidates.append((b - m0 * B[i], acc_dist + d0))

        if len(new_candidates) > max_list:
            new_candidates.sort(key=lambda x: x[1])
            candidates = new_candidates[:max_list]
        else:
            candidates = new_candidates

    best_b, _ = min(candidates, key=lambda x: np.dot(x[0], x[0]))
    return target - best_b, time.perf_counter() - start, stats

## Proposed Novel Algorithm: Adaptive-Slack Va-Anp (As-Va-Anp)

In [5]:
# ============================================================================
# 4. PROPOSED NOVEL ALGORITHM: ADAPTIVE-SLACK VA-ANP (AS-VA-ANP)
# ============================================================================

def proposed_as_va_anp(B, B_star, mu, nrm_sq, target, sigma, max_list=None):
    """5. PROPOSED Adaptive-Slack VA-ANP (AS-VA-ANP).
    Combines fast mu-space coordinate projection with a clean, lightweight
    closed-form adaptive penalty threshold that scales with local Gram-Schmidt
    geometry and remaining sphere budget—slashing unnecessary node expansions
    while maximizing exact recovery success.
    """
    start = time.perf_counter()
    n = B.shape[0]
    if max_list is None:
        max_list = 2 * n

    sigma_sq = sigma ** 2
    y_target = (B_star @ target) / nrm_sq

    # Dynamic global sphere bound scaling with n and sigma^2
    max_sphere_radius = 2.65 * n * sigma_sq

    candidates = [([], 0.0)]
    stats = {"nodes": 0, "children": 0}

    for i in range(n - 1, -1, -1):
        b_star_i = B_star[i]
        nsq = nrm_sq[i]

        # Closed-form adaptive penalty scaling inversely with local GS norm ratio
        local_ratio = nsq / (nsq + 3.0 * sigma_sq)
        dynamic_penalty = 2.2 * sigma_sq * (1.0 + 1.5 * (1.0 - local_ratio))

        new_candidates = []

        for m_hist, acc_dist in candidates:
            stats["nodes"] += 1

            c = y_target[i]
            for idx_step, m_prev in enumerate(m_hist):
                prev_dim = n - 1 - idx_step
                c -= m_prev * mu[prev_dim, i]

            se_planes = schnorr_euchner_planes(c, 3)
            d0 = ((c - se_planes[0]) ** 2) * nsq
            acc_0 = acc_dist + d0

            if acc_0 <= max_sphere_radius or not new_candidates:
                new_candidates.append((m_hist + [se_planes[0]], acc_0))
                stats["children"] += 1

            for m_k in se_planes[1:]:
                dk = ((c - m_k) ** 2) * nsq
                acc_k = acc_dist + dk
                delta_k = dk - d0

                if delta_k <= dynamic_penalty and acc_k <= max_sphere_radius:
                    new_candidates.append((m_hist + [m_k], acc_k))
                    stats["children"] += 1
                else:
                    break

        if not new_candidates:
            m_hist, acc_dist = candidates[0]
            c = y_target[i]
            for idx_step, m_prev in enumerate(m_hist):
                prev_dim = n - 1 - idx_step
                c -= m_prev * mu[prev_dim, i]
            m0 = int(round(c))
            d0 = ((c - m0) ** 2) * nsq
            new_candidates.append((m_hist + [m0], acc_dist + d0))

        if len(new_candidates) > max_list:
            new_candidates.sort(key=lambda x: x[1])
            candidates = new_candidates[:max_list]
        else:
            candidates = new_candidates

    best_hist, _ = min(candidates, key=lambda x: x[1])
    coeffs_asc = np.array(best_hist[::-1], dtype=float)
    best_lattice_vec = coeffs_asc @ B

    return best_lattice_vec, time.perf_counter() - start, stats

## Benchmark Runner

In [6]:
# ============================================================================
# 5. BENCHMARK RUNNER
# ============================================================================

def run_fast_benchmark():
    overall_start = time.perf_counter()
    print("=" * 96)
    print("  FAST & LEAN PRIMAL BDD BENCHMARK: AS-VA-ANP vs. BASELINES")
    print("=" * 96)

    configs = [(2, 4), (2, 6)]
    noise_ratios = [0.24, 0.30, 0.36]
    num_bases_per_dim = 4
    trials_per_basis = 5
    eta = 2

    algos = [
        "1. Babai Nearest Plane",
        "2. Lindner-Peikert ANP",
        "3. Discrete Pruned Enum (2023)",
        "4. Static VA-ANP (Heuristic)",
        "5. Proposed AS-VA-ANP (Optimized)"
    ]

    metrics = {a: {"succ": 0, "nodes": [], "time": [], "err": []} for a in algos}
    by_snr = {r: {a: 0 for a in algos} for r in noise_ratios}
    total_instances = len(configs) * len(noise_ratios) * num_bases_per_dim * trials_per_basis

    for k_rank, poly_deg in configs:
        dim = 2 * k_rank * poly_deg
        t_lll_start = time.perf_counter()
        reduced_bases = []
        for _ in range(num_bases_per_dim):
            B_raw = generate_mlkem_bdd_basis(k_rank=k_rank, poly_deg=poly_deg, q=3329)
            reduced_bases.append(fast_lll_reduce(B_raw))
        lll_ms = (time.perf_counter() - t_lll_start) * 1000.0 / num_bases_per_dim
        print(f"  [Dim n={dim:2d}] Fast Incremental LLL avg time per basis: {lll_ms:.2f} ms")

        for ratio in noise_ratios:
            for B_lll, B_star, mu, nrm_sq in reduced_bases:
                for _ in range(trials_per_basis):
                    target, true_pt, sigma_eff = generate_cbd_bdd_target(
                        B_lll, nrm_sq, eta=eta, ratio=ratio
                    )

                    runs = [
                        (algos[0], babai_nearest_plane(B_lll, B_star, nrm_sq, target)),
                        (algos[1], lindner_peikert_anp(B_lll, B_star, nrm_sq, target, sigma_eff)),
                        (algos[2], discrete_pruned_enumeration_2023(B_lll, B_star, nrm_sq, target)),
                        (algos[3], static_va_anp_heuristic(B_lll, B_star, nrm_sq, target, sigma_eff)),
                        (algos[4], proposed_as_va_anp(B_lll, B_star, mu, nrm_sq, target, sigma_eff)),
                    ]

                    for name, (v_hat, elapsed, st) in runs:
                        err = norm(v_hat - true_pt)
                        ok = int(err < 1e-3)
                        metrics[name]["succ"] += ok
                        metrics[name]["nodes"].append(st["children"])
                        metrics[name]["time"].append(elapsed * 1000.0)
                        metrics[name]["err"].append(err)
                        by_snr[ratio][name] += ok

    print("\n" + "=" * 96)
    print(f"  FINAL AGGREGATE COMPARISON ({total_instances} Module-LWE CBD_2 BDD Attack Instances)")
    print("=" * 96)
    print(f"  {'Algorithm':<32} {'Success Rate':>13} {'Avg Nodes':>12} {'Avg Time':>12} {'Efficiency (Succ/ms)':>21}")
    print("  " + "-" * 92)

    for name in algos:
        m = metrics[name]
        sr = (m["succ"] / total_instances) * 100.0
        nd = np.mean(m["nodes"])
        tm = np.mean(m["time"])
        eff = sr / max(tm, 1e-6)
        print(f"  {name:<32} {sr:>12.1f}% {nd:>12.1f} {tm:>10.3f}ms {eff:>20.1f}")

    print("  " + "-" * 92)
    print(f"  Total Benchmark Wall Time: {time.perf_counter() - overall_start:.2f} s")
    print("=" * 96)

## Run the benchmark

In [7]:
run_fast_benchmark()

  FAST & LEAN PRIMAL BDD BENCHMARK: AS-VA-ANP vs. BASELINES


  [Dim n=16] Fast Incremental LLL avg time per basis: 1223.15 ms


  [Dim n=24] Fast Incremental LLL avg time per basis: 8293.39 ms



  FINAL AGGREGATE COMPARISON (120 Module-LWE CBD_2 BDD Attack Instances)
  Algorithm                         Success Rate    Avg Nodes     Avg Time  Efficiency (Succ/ms)
  --------------------------------------------------------------------------------------------
  1. Babai Nearest Plane                   19.2%         20.0      0.296ms                 64.6
  2. Lindner-Peikert ANP                   37.5%       1429.1     18.138ms                  2.1
  3. Discrete Pruned Enum (2023)           32.5%        832.0     13.095ms                  2.5
  4. Static VA-ANP (Heuristic)             32.5%        704.5     11.319ms                  2.9
  5. Proposed AS-VA-ANP (Optimized)         32.5%        486.3      6.295ms                  5.2
  --------------------------------------------------------------------------------------------
  Total Benchmark Wall Time: 44.02 s


## Checks

Things the benchmark doesn't show:

1. **What the instances are.** `generate_cbd_bdd_target` takes a random lattice point (coefficients in −3..3) and adds
   real-valued noise scaled to `ratio` × the harmonic mean of the basis's Gram-Schmidt norms. There is no secret s and
   no b = A s + e mod q, and the noise is (CBD + 0.25·CBD) rescaled, not ML-KEM's CBD. Lattice dimensions are 16 and 24.
   So these are synthetic BDD instances on small module-structured lattices, not ML-KEM or LWE attack instances.
2. **Does "adaptive slack" do anything?** `dynamic_penalty = 2.2σ²·(1 + 1.5·(1 − ‖b*ᵢ‖²/(‖b*ᵢ‖² + 3σ²)))`, against
   the static rule's `llr_threshold · 2σ²`. The cell below measures how much the factor in brackets varies from level
   to level, and runs static VA-ANP with the bare constants (1.1, 2.65) on the same instances. The fair question is
   whether AS-VA-ANP beats a static setting that spends the same number of nodes; that is the last test.
3. **Sample size.** 120 instances give 95% intervals of about ±9 points; the larger run below uses 3,000.

In [8]:
import math


def wilson95(k, n, z=1.959963984540054):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return 100 * max(0.0, c - h), 100 * min(1.0, c + h)


def mcnemar_p(a, b):
    """Exact two-sided McNemar p-value for paired success lists a, b."""
    x = sum(1 for u, v in zip(a, b) if u and not v)
    y = sum(1 for u, v in zip(a, b) if v and not u)
    if x + y == 0:
        return x, y, 1.0
    tail = sum(math.comb(x + y, k) for k in range(min(x, y) + 1)) / 2 ** (x + y)
    return x, y, min(1.0, 2 * tail)


def compare(variants, seed=42, configs=((2, 4), (2, 6)), ratios=(0.24, 0.30, 0.36), bases=4, trials=5, eta=2):
    """Same instance generator and order as run_fast_benchmark (the decoders draw no random numbers),
    so with the default sizes the instances are identical to the benchmark's. Also records how far the
    'adaptive' penalty moves away from 2.2 sigma^2 at each level."""
    np.random.seed(seed)
    got = {k: [] for k in variants}
    nodes = {k: [] for k in variants}
    factor = []  # dynamic_penalty / (2.2 sigma^2) per level
    for k_rank, poly_deg in configs:
        red = [fast_lll_reduce(generate_mlkem_bdd_basis(k_rank=k_rank, poly_deg=poly_deg, q=3329)) for _ in range(bases)]
        for ratio in ratios:
            for B_lll, B_star, mu, nrm_sq in red:
                for _ in range(trials):
                    target, true_pt, sigma_eff = generate_cbd_bdd_target(B_lll, nrm_sq, eta=eta, ratio=ratio)
                    lr = nrm_sq / (nrm_sq + 3.0 * sigma_eff ** 2)
                    factor.extend(1.0 + 1.5 * (1.0 - lr))
                    for name, f in variants.items():
                        v_hat, _, st = f(B_lll, B_star, mu, nrm_sq, target, sigma_eff)
                        got[name].append(bool(norm(v_hat - true_pt) < 1e-3))
                        nodes[name].append(st["children"])
    N = len(next(iter(got.values())))
    print(f"{N} instances. Success with 95% Wilson interval; nodes = children expanded (the benchmark's 'Avg Nodes').")
    for name in variants:
        lo, hi = wilson95(sum(got[name]), N)
        print(f"  {name:<44} {100 * np.mean(got[name]):5.1f}%  ({lo:4.1f}-{hi:4.1f})  nodes {np.mean(nodes[name]):7.1f}")
    factor = np.array(factor)
    print(f"'Adaptive' penalty / (2.2 sigma^2): median {np.median(factor):.3f}, 90th pct {np.percentile(factor, 90):.3f}, "
          f"max {factor.max():.3f}; share of levels within 1% of constant: {100 * np.mean(factor < 1.01):.1f}%")
    return got, nodes


VARIANTS = {
    "Proposed AS-VA-ANP (as given)": lambda B, Bs, mu, nrm, t, s: proposed_as_va_anp(B, Bs, mu, nrm, t, s),
    "Static VA-ANP 2.2 / 2.8 (as given)": lambda B, Bs, mu, nrm, t, s: static_va_anp_heuristic(B, Bs, nrm, t, s),
    "Static VA-ANP 1.1 / 2.65 (AS constants)": lambda B, Bs, mu, nrm, t, s: static_va_anp_heuristic(
        B, Bs, nrm, t, s, llr_threshold=1.1, sphere_mult=2.65),
    "Lindner-Peikert ANP (as given)": lambda B, Bs, mu, nrm, t, s: lindner_peikert_anp(B, Bs, nrm, t, s),
}

### Same 120 instances as the benchmark

In [9]:
got, nodes = compare(VARIANTS)

120 instances. Success with 95% Wilson interval; nodes = children expanded (the benchmark's 'Avg Nodes').
  Proposed AS-VA-ANP (as given)                 32.5%  (24.8-41.3)  nodes   486.3
  Static VA-ANP 2.2 / 2.8 (as given)            32.5%  (24.8-41.3)  nodes   704.5
  Static VA-ANP 1.1 / 2.65 (AS constants)       29.2%  (21.8-37.8)  nodes   338.9
  Lindner-Peikert ANP (as given)                37.5%  (29.4-46.4)  nodes  1429.1
'Adaptive' penalty / (2.2 sigma^2): median 1.301, 90th pct 1.503, max 1.697; share of levels within 1% of constant: 0.0%


### Paired test on those instances (exact McNemar)

In [10]:
ref = "Proposed AS-VA-ANP (as given)"
for name in VARIANTS:
    if name != ref:
        x, y, p = mcnemar_p(got[ref], got[name])
        print(f"AS-VA-ANP vs {name:<40} only AS {x:3d}, only other {y:3d}, p = {p:.3g}")

AS-VA-ANP vs Static VA-ANP 2.2 / 2.8 (as given)       only AS   0, only other   0, p = 1
AS-VA-ANP vs Static VA-ANP 1.1 / 2.65 (AS constants)  only AS   4, only other   0, p = 0.125
AS-VA-ANP vs Lindner-Peikert ANP (as given)           only AS   0, only other   6, p = 0.0312


### Larger run: 20 bases × 25 targets per (dimension, ratio), 3,000 instances

In [11]:
big_got, big_nodes = compare(VARIANTS, seed=7, bases=20, trials=25)
ref = "Proposed AS-VA-ANP (as given)"
for name in VARIANTS:
    if name != ref:
        x, y, p = mcnemar_p(big_got[ref], big_got[name])
        print(f"AS-VA-ANP vs {name:<40} only AS {x:3d}, only other {y:3d}, p = {p:.3g}")

3000 instances. Success with 95% Wilson interval; nodes = children expanded (the benchmark's 'Avg Nodes').
  Proposed AS-VA-ANP (as given)                 31.8%  (30.2-33.5)  nodes   492.8
  Static VA-ANP 2.2 / 2.8 (as given)            34.7%  (33.1-36.5)  nodes   703.6
  Static VA-ANP 1.1 / 2.65 (AS constants)       29.5%  (27.9-31.2)  nodes   322.1
  Lindner-Peikert ANP (as given)                37.1%  (35.4-38.8)  nodes  1422.0
'Adaptive' penalty / (2.2 sigma^2): median 1.303, 90th pct 1.500, max 1.723; share of levels within 1% of constant: 0.0%
AS-VA-ANP vs Static VA-ANP 2.2 / 2.8 (as given)       only AS  25, only other 112, p = 2.4e-14
AS-VA-ANP vs Static VA-ANP 1.1 / 2.65 (AS constants)  only AS  92, only other  23, p = 5.79e-11
AS-VA-ANP vs Lindner-Peikert ANP (as given)           only AS  38, only other 196, p = 7.65e-27


### Equal-node comparison

AS-VA-ANP uses fewer nodes than static 2.2 / 2.8 and more than static 1.1 / 2.65, so its success should be compared with a static setting that spends the **same** number of nodes. The sweep below varies the static `llr_threshold` (sphere 2.8) on the same 3,000 instances; the setting closest in nodes is then compared with AS-VA-ANP by McNemar's test.

In [12]:
sweep = {f"Static VA-ANP {l} / 2.8": (lambda l: lambda B, Bs, mu, nrm, t, s: static_va_anp_heuristic(
    B, Bs, nrm, t, s, llr_threshold=l, sphere_mult=2.8))(l) for l in (1.2, 1.35, 1.5, 1.65)}
sw_got, sw_nodes = compare(sweep, seed=7, bases=20, trials=25)
ref_nodes = np.mean(big_nodes["Proposed AS-VA-ANP (as given)"])
match = min(sweep, key=lambda k: abs(np.mean(sw_nodes[k]) - ref_nodes))
x, y, p = mcnemar_p(big_got["Proposed AS-VA-ANP (as given)"], sw_got[match])
print(f"\nAS-VA-ANP: {100 * np.mean(big_got['Proposed AS-VA-ANP (as given)']):.1f}% at {ref_nodes:.0f} nodes; "
      f"{match}: {100 * np.mean(sw_got[match]):.1f}% at {np.mean(sw_nodes[match]):.0f} nodes; "
      f"only AS {x}, only static {y}, p = {p:.3g}")

3000 instances. Success with 95% Wilson interval; nodes = children expanded (the benchmark's 'Avg Nodes').
  Static VA-ANP 1.2 / 2.8                       30.3%  (28.7-32.0)  nodes   364.5
  Static VA-ANP 1.35 / 2.8                      31.3%  (29.7-33.0)  nodes   423.3
  Static VA-ANP 1.5 / 2.8                       32.1%  (30.5-33.8)  nodes   479.5
  Static VA-ANP 1.65 / 2.8                      32.9%  (31.3-34.6)  nodes   532.2
'Adaptive' penalty / (2.2 sigma^2): median 1.303, 90th pct 1.500, max 1.723; share of levels within 1% of constant: 0.0%

AS-VA-ANP: 31.8% at 493 nodes; Static VA-ANP 1.5 / 2.8: 32.1% at 480 nodes; only AS 9, only static 18, p = 0.122


## Findings (seeded, so the numbers above reproduce exactly)

- **It runs** once the `__main__` line is fixed. On its own 120 instances AS-VA-ANP, static VA-ANP and the
  "2023 enumeration" all recover 32.5%; 120 instances give ±9-point intervals, too few to rank anything.
- **The adaptive penalty does vary** from level to level (median 1.30×, up to 1.7× the base 2.2σ²).
- **But it buys nothing over static VA-ANP.** On 3,000 instances AS-VA-ANP recovers 31.8% at 493 nodes. Static
  VA-ANP with `llr_threshold = 1.5` recovers 32.1% at 480 nodes on the same instances (McNemar p = 0.12, static
  slightly ahead). The "adaptive slack" gives the same success-vs-nodes trade-off as retuning one static constant;
  the node saving in the benchmark table comes from tighter constants, at the cost of 2.9 points of success against
  static 2.2 / 2.8 (p = 2·10⁻¹⁴).
- **The instances are not ML-KEM.** The output's title says "Module-LWE CBD_2 BDD attack instances", but there is no
  LWE secret, the noise is rescaled real-valued (CBD + 0.25·CBD), and the dimensions are 16 and 24.
- **The named baselines are not the published algorithms.** "Lindner-Peikert" here caps planes at 3 per level and
  cuts the list to 2n; "Discrete Pruned Enum (2023)" is a fixed set of 0/1 tags, not Luan et al.'s cell enumeration.
- **"Efficiency (Succ/ms)" mixes success with Python run time** of different implementations; use success at equal
  node count instead, as above.